# Retina Age — session 1: baselines and regularization isolation

**Settings (right-hand panel):**
- Accelerator: **GPU T4**
- Internet: Off is fine — everything comes from attached data

**Data sources to attach:**
1. the output of the `01_prepare_data` notebook (splits + images_224)
2. the `retina-code` dataset (our `src/` folder)

This session runs 5 configurations x 3 seeds = 15 runs, about 75 minutes.


## 1. Environment check


In [ ]:
import torch, os, sys
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('CPUs:', os.cpu_count())
!ls /kaggle/input


## 2. Put the code on the path

`src/` is attached read-only, so it is copied into the working directory - that also
means `results/` lands somewhere writable and gets saved as this notebook's output.


In [ ]:
import shutil
from pathlib import Path

# find the attached folder that contains src/
code_src = next(p for p in Path('/kaggle/input').rglob('src') if (p/'train.py').exists())
print('found code at:', code_src)

work = Path('/kaggle/working/project')
work.mkdir(parents=True, exist_ok=True)
shutil.copytree(code_src, work/'src', dirs_exist_ok=True)
sys.path.insert(0, str(work))

for m in [k for k in list(sys.modules) if k == 'src' or k.startswith('src.')]:
    del sys.modules[m]
print('ok')


## 3. Blocking checks

Patient overlap must be zero in all three pairs, or nothing below means anything.
The mean predictor is the bar every model has to clear - it belongs at the top of
the results table.


In [ ]:
from src.train import check_patient_leakage, run_mean_predictor
check_patient_leakage()
run_mean_predictor()


## 4. Non-deep baseline

Ridge regression on hand-crafted features. Deterministic, so one seed is enough.


In [ ]:
from src.train import train_linear_baseline
train_linear_baseline()


## 5. Baseline CNN and the regularization isolation

Each of these differs from `cnn_plain` in exactly one factor, which is the whole
point - our first attempt changed six things at once and the comparison meant nothing.

| config | one factor vs cnn_plain |
|---|---|
| `cnn_dropout` | dropout |
| `cnn_wd` | weight decay |
| `cnn_es` | early stopping (best-val weights instead of final) |
| `cnn_all` | all three together |


In [ ]:
from src.train import run_experiment

for name in ['cnn_plain', 'cnn_dropout', 'cnn_wd', 'cnn_es', 'cnn_all']:
    run_experiment(name, seeds=[0, 1, 2])


## 6. Results so far


In [ ]:
from src.train import aggregate_results
aggregate_results()


## 7. Save

**Save Version** writes `/kaggle/working/` as the notebook output, so `results/`
survives with every metrics file, figure and checkpoint. Session 2 attaches this
output and picks up where this left off - anything already done is skipped.


In [ ]:
import shutil
shutil.make_archive('/kaggle/working/results', 'zip', '/kaggle/working/project/results')
print('results.zip written -', round(Path('/kaggle/working/results.zip').stat().st_size/1e6, 1), 'MB')
